In [1]:
import contextily as cx
import folium
import geopandas as gpd
import numpy as np
import pandas as pd

rng = np.random.default_rng(seed=42)

import altair as alt
import attaviz

attaviz.enable()

TILES = cx.providers.Esri.WorldGrayCanvas
THRESHOLDS = [15, 30, 45]
LOCAL_CRS = "EPSG:32721"
WGS84_CRS = "EPSG:4326"


def simplify_for_web(frame, tolerance=30):
    display_frame = frame.to_crs(LOCAL_CRS).copy()
    display_frame.geometry = display_frame.geometry.simplify(
        tolerance, preserve_topology=True
    )
    return display_frame.to_crs(WGS84_CRS)


In [2]:
data = pd.read_csv("../data/kido/od_movementType_personales.csv")
personas = pd.read_csv(
    "../data/population/personas_ext_07_2026.csv",
    encoding="latin-1",
    dtype={
        "ID_CENSO": "str",
        "DEPARTAMENTO": "str",
        "SEGMENTO": "str",
        "SECCION": "str",
    },
)

/var/folders/q1/wt8mfyzs73l2r5977rk_mkxm0000gn/T/ipykernel_6201/1990546031.py:2: DtypeWarning: Columns (14: BARRIO85, 151: ESTRATO) have mixed types. Specify dtype option on import or set low_memory=False.
  personas = pd.read_csv(


In [3]:
zonas = gpd.read_file("../data/geography-levels/zonas-a15-agg.gpkg").set_index(
    "poligono_id"
)

# Análisis de la matriz origen-destino (OD)

A continuación se presentan los resultados del análisis de la matriz origen-destino (OD) elaborada por KidoDynamics a partir de datos CDR (*Call Detail Records*) y xDR (*x Data Records*) de Claro Uruguay.

**Zonificación**

La zonificación utilizada en este análisis fue elaborada por el World Bank a partir de los centroides de un estudio previo de Steer. Como la zonificación original era muy detallada, el equipo la agrupó para obtener zonas de mayor extensión y adecuarla al análisis.

Para más información sobre el proceso, consulte estas notebooks:
- [Recreación de las zonas del estudio de Steer a partir de sus centroides](recreacion-zonas-steer)
- [Agregación de la zonificación en zonas más grandes](agregar-zonas)
- [Agregación de la zonificación según población](zonas-agg-poblacion)

La zonificación se dividió en Core y Perfiférica. Esto significa que para las zonas Periféricas no se informará  
**ODs**

Por el momento, se dispone de viajes entre pares origen-destino, clasificados como *Trips* y *MicroTrips*:
- **Trips**: movimientos que recorren al menos 5 km en un período máximo de 1 hora. Generalmente se asocian con modos de transporte motorizados.
- **MicroTrips**: movimientos de corta distancia y duración relativamente larga. Generalmente se asocian con desplazamientos a pie o recorridos muy cortos.


Para representar los viajes personales y excluir los desplazamientos de conductores profesionales, se excluyeron de la matriz las personas cuya distancia media diaria superaba los 100 km.

**Días**

8, 15 y 22 de Octubre del 2025



## Imputación valores sustitutos. 
KidoDynamics no puede publicar datos correspondientes a celdas con menos de 10 viajes. En esta matriz, cada par origen-destino (OD), desagregado por tipo de viaje (Trip o Microtrip), que registra menos de 10 viajes se informa como `"<10"`. Para estimar esos valores, sustituimos cada `"<10"` por un entero aleatorio entre 1 y 9, ambos inclusive.

A continuación, calculamos cuánto varía, en porcentaje, la suma total de viajes tras la imputación respecto de la suma de los viajes publicados.

In [4]:
viajes_originales = (
    data[data["total_trips"] != "<10"]["total_trips"].astype(float).sum() / 3
)

In [5]:
mask_menos_10 = data["total_trips"].eq("<10")
data["total_trips"] = pd.to_numeric(data["total_trips"], errors="coerce")
data.loc[mask_menos_10, "total_trips"] = rng.integers(1, 10, size=mask_menos_10.sum())

viajes_totales = int(np.round(data["total_trips"].sum() / 3, 0))

In [6]:
print("Error: ", round(((viajes_totales / viajes_originales) - 1) * 100, 2), "%")

Error:  0.82 %


In [7]:
data["total_trips"] = data["total_trips"].astype(float) / 3

## Viajes por persona 
Se calcula el promedio de viajes por persona para la población de 18 años o más.

```{note}
Se utiliza este umbral de edad porque KidoDynamics solo dispone de datos para personas de 18 años o más.
```

```{note}
El cálculo no incluye los viajes que tienen lugar dentro de las zonas periféricas, por lo que el promedio de viajes por persona podría estar subestimado.
```

La tasa de generación de viajes por persona es superior a la estimada en la Encuesta de Movilidad de 2017. Esta diferencia podría deberse, entre otros factores, a que:

- La matriz OD no distingue las distintas etapas de un mismo viaje.
- Los datos de telefonía pueden registrar desplazamientos cortos que suelen estar subrepresentados en las encuestas de movilidad.

In [8]:
personas["CODSEG"] = (
    personas["DEPARTAMENTO"] + personas["SECCION"] + personas["SEGMENTO"]
)

/var/folders/q1/wt8mfyzs73l2r5977rk_mkxm0000gn/T/ipykernel_6201/3301231239.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  personas["CODSEG"] = (


In [9]:
cods_aoi = []
for index, row in zonas[["CODSEG"]].iterrows():
    cods_aoi += row["CODSEG"].split(", ")

In [10]:
mayores_18 = int(
    np.round(
        personas[
            (personas["PERNA01"] >= 18) & (personas["CODSEG"].isin(cods_aoi))
        ].W.sum(),
        0,
    )
)

In [11]:
df = pd.DataFrame(
    index=["Personas", "Viajes"],
    columns=[
        "Totales",
        "Trips",
        "MicroTrips",
        "EM2017",
        "EM2017 (Considerando Etapas)",
    ],
)
df.loc["Personas", "Totales"] = mayores_18
df.loc["Viajes", "Totales"] = viajes_totales
df.loc["Viajes por persona", "Totales"] = np.round(
    df.loc["Viajes", "Totales"] / df.loc["Personas", "Totales"], 2
)

df.loc["Personas", "Trips"] = mayores_18
df.loc["Viajes", "Trips"] = round(
    data[data["movement_type"] == "trip"]["total_trips"].sum(), 2
)
df.loc["Viajes por persona", "Trips"] = np.round(
    df.loc["Viajes", "Trips"] / df.loc["Personas", "Trips"], 2
)

df.loc["Personas", "MicroTrips"] = mayores_18
df.loc["Viajes", "MicroTrips"] = round(
    data[data["movement_type"] == "utrip"]["total_trips"].sum(), 2
)
df.loc["Viajes por persona", "MicroTrips"] = np.round(
    df.loc["Viajes", "MicroTrips"] / df.loc["Personas", "MicroTrips"], 2
)

df.loc["Personas", "EM2017"] = 1806989
df.loc["Viajes", "EM2017"] = 4201184
df.loc["Viajes por persona", "EM2017"] = np.round(
    df.loc["Viajes", "EM2017"] / df.loc["Personas", "EM2017"], 2
)

df.loc["Personas", "EM2017 (Considerando Etapas)"] = 1806989
df.loc["Viajes", "EM2017 (Considerando Etapas)"] = 4729907
df.loc["Viajes por persona", "EM2017 (Considerando Etapas)"] = np.round(
    df.loc["Viajes", "EM2017 (Considerando Etapas)"]
    / df.loc["Personas", "EM2017 (Considerando Etapas)"],
    2,
)

df


,Totales,Trips,MicroTrips,EM2017,EM2017 (Considerando Etapas)
Personas,2110225,2110225,2110225,1806989,1806989
Viajes,6067486,2651390.02,3416096.3,4201184,4729907
Viajes por persona,2.88,1.26,1.62,2.32,2.62


## Viajes Atraídos
El siguiente mapa muestra los viajes que atrae cada una de las zonas. Las diferentes capas muestran:
- Viajes atraídos totales
- Viajes atraídos trips
- Viajes atraídos microtrips
- Ratio entre viajes atraídos microtrips/trips. La idea de esto último es ver si existen zonas que atraen más viajes de movilidad activa. 

```{caution}
Se excluyen de este análisis los viajes que ocurren dentro de la zona misma. 
```

**Conclusiones**
- Observamos que la zona de la terminal de tres cruces es la que más viajes atrae, tanto de Trips como MictroTrips. 
- Tanto en Ciudad Vieja como en Pocitos, el ratio de MictroTrips/Trips es más elevado que la media (1.3)

In [12]:
zonas["Viajes Atraídos"] = (
    data[data["origin"] != data["destination"]]
    .groupby("destination")
    .total_trips.sum()
    .round(0)
    .astype("int")
)
zonas["Viajes Atraídos Trip"] = (
    data[(data["origin"] != data["destination"]) & (data["movement_type"] == "trip")]
    .groupby("destination")
    .total_trips.sum()
    .round(0)
    .astype("int")
)
zonas["Viajes Atraídos MicroTrip"] = (
    data[(data["origin"] != data["destination"]) & (data["movement_type"] == "utrip")]
    .groupby("destination")
    .total_trips.sum()
    .round(0)
    .astype("int")
)
zonas["Ratio MicroTrip/Trip (Atraídos)"] = (
    zonas["Viajes Atraídos MicroTrip"] / zonas["Viajes Atraídos Trip"]
)


In [13]:
cols = [
    "geometry",
    "poblacion",
    "Viajes Atraídos",
    "Viajes Atraídos Trip",
    "Viajes Atraídos MicroTrip",
    "Ratio MicroTrip/Trip (Atraídos)",
]
m = simplify_for_web(zonas[zonas["tipo_zona"] == "Core"][cols]).explore(
    column="Viajes Atraídos",
    cmap="Spectral_r",
    name="Viajes Atraídos",
    zoom_control=False,
    tiles=TILES,
)
simplify_for_web(zonas[zonas["tipo_zona"] == "Core"][cols]).explore(
    column="Viajes Atraídos Trip", cmap="Spectral_r", m=m, name="Viajes Atraídos Trip"
)
simplify_for_web(zonas[zonas["tipo_zona"] == "Core"][cols]).explore(
    column="Viajes Atraídos MicroTrip",
    cmap="Spectral_r",
    m=m,
    name="Viajes Atraídos MicroTrip",
)

simplify_for_web(zonas[zonas["tipo_zona"] == "Core"][cols]).explore(
    column="Ratio MicroTrip/Trip (Atraídos)",
    cmap="Spectral_r",
    m=m,
    name="Ratio MicroTrip/Trip",
)

folium.LayerControl().add_to(m)
m

### Viajes Atraídos por persona por sección censal
El mapa de abajo agrupa las variables anteriores por sección censal y las estandariza por el número de personas que viven en la sección, excepto por el ratio. 

Al realizar la estandarización por población observamos que las zonas que más atraen viajes son el Centro, Ciudad Vieja y Aguada.

In [14]:
sc = gpd.read_file("../data/geography-levels/sc_23_pg.gpkg").set_index("CODSECC")
sc_core = zonas[
    zonas["tipo_zona"] == "Core"
].seccion_codigo  # en la seccion core cada zona pertenece a una seccion.
sc = sc[(sc.index.isin(sc_core.unique())) & (sc.index != "0125")].copy()
sc["poblacion"] = zonas.groupby("seccion_codigo")["poblacion"].sum()
sc.loc["1600", "poblacion"] = 2
for tipo in ["Viajes Atraídos", "Viajes Atraídos Trip", "Viajes Atraídos MicroTrip"]:
    sc[tipo] = zonas.groupby("seccion_codigo")[tipo].sum() / sc["poblacion"]
sc["Ratio MicroTrip/Trip (Atraídos)"] = (
    sc["Viajes Atraídos MicroTrip"] / sc["Viajes Atraídos Trip"]
)

In [15]:
cols = [
    "geometry",
    "Viajes Atraídos",
    "Viajes Atraídos Trip",
    "Viajes Atraídos MicroTrip",
    "Ratio MicroTrip/Trip (Atraídos)",
]
m = simplify_for_web(sc[cols]).explore(
    column="Viajes Atraídos",
    cmap="Spectral_r",
    name="Viajes Atraídos por persona",
    zoom_control=False,
    tiles=TILES,
)

simplify_for_web(sc[cols]).explore(
    column="Viajes Atraídos Trip",
    cmap="Spectral_r",
    m=m,
    name="Viajes Atraídos Trip por persona",
)
simplify_for_web(sc[cols]).explore(
    column="Viajes Atraídos MicroTrip",
    cmap="Spectral_r",
    m=m,
    name="Viajes Atraídos MicroTrip por persona",
)

simplify_for_web(sc[cols]).explore(
    column="Ratio MicroTrip/Trip (Atraídos)",
    cmap="Spectral_r",
    m=m,
    name="Ratio MicroTrip/Trip por persona",
)

folium.LayerControl().add_to(m)
m

## Viajes Generados
El siguiente mapa muestra los viajes que genera cada una de las zonas. Las diferentes capas muestran:
- Viajes generados totales
- Viajes generados trips
- Viajes generados microtrips
- Ratio entre viajes generados microtrips/trips. La idea de esto último es ver si existen zonas que genera más viajes de movilidad activa. 

```{caution}
Se excluyen de este análisis los viajes que ocurren dentro de la zona misma. 
```

**Conclusiones**
La distribución espacial es similar a la de los viajes atraídos. 

In [16]:
zonas["Viajes Generados"] = (
    data[data["origin"] != data["destination"]]
    .groupby("origin")
    .total_trips.sum()
    .round(0)
    .astype("int")
)
zonas["Viajes Generados Trip"] = (
    data[(data["origin"] != data["destination"]) & (data["movement_type"] == "trip")]
    .groupby("origin")
    .total_trips.sum()
    .round(0)
    .astype("int")
)
zonas["Viajes Generados MicroTrip"] = (
    data[(data["origin"] != data["destination"]) & (data["movement_type"] == "utrip")]
    .groupby("origin")
    .total_trips.sum()
    .round(0)
    .astype("int")
)
zonas["Ratio MicroTrip/Trip (Generados)"] = (
    zonas["Viajes Generados MicroTrip"] / zonas["Viajes Generados Trip"]
)

In [17]:
cols = [
    "geometry",
    "poblacion",
    "Viajes Generados",
    "Viajes Generados Trip",
    "Viajes Generados MicroTrip",
    "Ratio MicroTrip/Trip (Generados)",
]
m = simplify_for_web(zonas[zonas["tipo_zona"] == "Core"][cols]).explore(
    column="Viajes Generados",
    cmap="Spectral_r",
    name="Viajes Generados",
    zoom_control=False,
    tiles=TILES,
)
simplify_for_web(zonas[zonas["tipo_zona"] == "Core"][cols]).explore(
    column="Viajes Generados Trip", cmap="Spectral_r", m=m, name="Viajes Generados Trip"
)
simplify_for_web(zonas[zonas["tipo_zona"] == "Core"][cols]).explore(
    column="Viajes Generados MicroTrip",
    cmap="Spectral_r",
    m=m,
    name="Viajes Generados MicroTrip",
)

simplify_for_web(zonas[zonas["tipo_zona"] == "Core"][cols]).explore(
    column="Ratio MicroTrip/Trip (Generados)",
    cmap="Spectral_r",
    m=m,
    name="Ratio MicroTrip/Trip",
)
folium.LayerControl().add_to(m)
# m

### Viajes generados por persona por sección censal
El mapa de abajo agrupa las variables anteriores por sección censal y las estandariza por el número de personas que viven en la sección, excepto por el ratio. 

Al igual que el caso anterior, la distribución espacial de viajes generados es similar a la de los viajes atraídos.

In [18]:
for tipo in ["Viajes Generados", "Viajes Generados Trip", "Viajes Generados MicroTrip"]:
    sc[tipo] = zonas.groupby("seccion_codigo")[tipo].sum() / sc["poblacion"]
sc["Ratio MicroTrip/Trip (Generados)"] = (
    sc["Viajes Generados MicroTrip"] / sc["Viajes Generados Trip"]
)

In [19]:
cols = [
    "geometry",
    "Viajes Generados",
    "Viajes Generados Trip",
    "Viajes Generados MicroTrip",
    "Ratio MicroTrip/Trip (Generados)",
]
m = simplify_for_web(sc[cols]).explore(
    column="Viajes Generados",
    cmap="Spectral_r",
    name="Viajes Generados por persona",
    zoom_control=False,
    tiles=TILES,
)

simplify_for_web(sc[cols]).explore(
    column="Viajes Generados Trip",
    cmap="Spectral_r",
    m=m,
    name="Viajes Generados Trip por persona",
)
simplify_for_web(sc[cols]).explore(
    column="Viajes Generados MicroTrip",
    cmap="Spectral_r",
    m=m,
    name="Viajes Generados MicroTrip por persona",
)

simplify_for_web(sc[cols]).explore(
    column="Ratio MicroTrip/Trip (Generados)",
    cmap="Spectral_r",
    m=m,
    name="Ratio MicroTrip/Trip por persona",
)

folium.LayerControl().add_to(m)
# m

## Viajes dentro de la zona
El siguiente mapa muestra los viajes que comienzan y terminan dentro de cada zona. Las diferentes capas muestran:
- Viajes totales
- Viajes trips
- Viajes microtrips
- Ratio entre viajes microtrips/trips. La idea de esto último es ver si existen zonas que genera más viajes de movilidad activa. 

**Conclusiones**
- El 44% de los viajes que comienzan y terminan dentro de una misma zona son microtrips contra el 20% que son trips. Esto tiene sentido puesto que una gran parte del área de estudio es el centro de Montevideo donde además de esperarse más viajes caminando, las zonas son de menor extensión. 
- En valores absolutos, la zona que contiene a Canelones y Juanicó se destaca por la cantidad de viajes que encierra. Sin embargo, cuando estandarizamos utilizando la población de la zona, esto ya no ocurre y los valores se acercan a los esperados. 

In [20]:
data[(data["origin"] == data["destination"]) & (data["movement_type"] == "utrip")][
    "total_trips"
].sum() / data[(data["movement_type"] == "utrip")]["total_trips"].sum()

np.float64(0.4397218081549966)

In [21]:
data[(data["origin"] == data["destination"]) & (data["movement_type"] == "trip")][
    "total_trips"
].sum() / data[(data["movement_type"] == "trip")]["total_trips"].sum()

np.float64(0.20048816607714298)

In [22]:
zonas["Viajes Dentro de la Zona"] = (
    data[data["origin"] == data["destination"]]
    .groupby("origin")
    .total_trips.sum()
    .round(0)
    .astype("int")
)
zonas["Viajes Dentro Trip"] = (
    data[(data["origin"] == data["destination"]) & (data["movement_type"] == "trip")]
    .groupby("origin")
    .total_trips.sum()
    .round(0)
    .astype("int")
)
zonas["Viajes Dentro MicroTrip"] = (
    data[(data["origin"] == data["destination"]) & (data["movement_type"] == "utrip")]
    .groupby("origin")
    .total_trips.sum()
    .round(0)
    .astype("int")
)
zonas["Ratio MicroTrip/Trip (Dentro)"] = (
    zonas["Viajes Dentro MicroTrip"] / zonas["Viajes Dentro Trip"]
)

In [23]:
cols = [
    "geometry",
    "poblacion",
    "Viajes Dentro de la Zona",
    "Viajes Dentro Trip",
    "Viajes Dentro MicroTrip",
    "Ratio MicroTrip/Trip (Dentro)",
]
m = simplify_for_web(zonas[zonas["tipo_zona"] == "Core"][cols]).explore(
    column="Viajes Dentro de la Zona",
    cmap="Spectral_r",
    name="Viajes Dentro de la Zona",
    zoom_control=False,
    tiles=TILES,
)
simplify_for_web(zonas[zonas["tipo_zona"] == "Core"][cols]).explore(
    column="Viajes Dentro Trip", cmap="Spectral_r", m=m, name="Viajes Dentro Trip"
)
simplify_for_web(zonas[zonas["tipo_zona"] == "Core"][cols]).explore(
    column="Viajes Dentro MicroTrip",
    cmap="Spectral_r",
    m=m,
    name="Viajes Dentro MicroTrip",
)
simplify_for_web(zonas[zonas["tipo_zona"] == "Core"][cols]).explore(
    column="Ratio MicroTrip/Trip (Dentro)",
    cmap="Spectral_r",
    m=m,
    name="Ratio MicroTrip/Trip",
)

folium.LayerControl().add_to(m)
# m

In [24]:
zonas["Viajes Dentro de la Zona por persona"] = (
    zonas["Viajes Dentro de la Zona"] / zonas["poblacion"]
)
zonas["Viajes Dentro Trip por persona"] = (
    zonas["Viajes Dentro Trip"] / zonas["poblacion"]
)
zonas["Viajes Dentro MicroTrip por persona"] = (
    zonas["Viajes Dentro MicroTrip"] / zonas["poblacion"]
)

In [25]:
cols = [
    "geometry",
    "poblacion",
    "Viajes Dentro de la Zona por persona",
    "Viajes Dentro Trip por persona",
    "Viajes Dentro MicroTrip por persona",
]
m = simplify_for_web(zonas[zonas["tipo_zona"] == "Core"][cols]).explore(
    column="Viajes Dentro de la Zona por persona",
    cmap="Spectral_r",
    name="Viajes Dentro de la Zona por persona",
    zoom_control=False,
    tiles=TILES,
)

simplify_for_web(zonas[zonas["tipo_zona"] == "Core"][cols]).explore(
    column="Viajes Dentro Trip por persona",
    cmap="Spectral_r",
    m=m,
    name="Viajes Dentro Trip por persona",
)
simplify_for_web(zonas[zonas["tipo_zona"] == "Core"][cols]).explore(
    column="Viajes Dentro MicroTrip por persona",
    cmap="Spectral_r",
    m=m,
    name="Viajes Dentro MicroTrip por persona",
)

folium.LayerControl().add_to(m)
# m

## Tiempos de viaje en Transporte Público
En esta sección estimamos el tiempo de viaje en transporte público para cada par origen-destino (OD) cuyos viajes comienzan y terminan en la zona Core. Para el cálculo se utilizan los GTFS que ya se describieron en el [análisis de accesibilidad](accessibility-jobs).

A diferencia del [análisis de accesibilidad](accessibility-jobs), que calcula cuántos puestos de trabajo pueden alcanzarse en determinado tiempo sin considerar si las personas cuentan con las habilidades requeridas para ocuparlos, este análisis se centra en los viajes efectivamente observados y estima cuánto habría tomado realizarlos en transporte público.

Una limitación importante es que no se considera la hora de salida de cada viaje, ya que todavía no contamos con matrices para distintos horarios. Por eso, los tiempos se calculan para un miércoles entre las 8:00 y las 9:00.

In [26]:
df = pd.read_parquet("../data/opportunities/wed_08-Za15-agg.parquet")

In [27]:
ids_core = zonas[zonas["tipo_zona"] == "Core"].index.unique()

In [28]:
ods_tt = (
    data[(data["origin"].isin(ids_core)) & (data["destination"].isin(ids_core))]
    .merge(
        df[["from_id", "to_id", "p50"]],
        how="left",
        left_on=["origin", "destination"],
        right_on=["from_id", "to_id"],
        validate="many_to_one",
    )
    .drop(columns=["from_id", "to_id"])
)
ods_tt["p50"] = ods_tt["p50"].fillna(180)

In [29]:
tiempos = ods_tt[["p50", "total_trips", "movement_type"]].copy()
tiempos = tiempos[(tiempos["p50"] >= 0) & (tiempos["total_trips"] > 0)].sort_values(
    "p50"
)

if tiempos.empty:
    raise ValueError("No hay tiempos y cantidades de viajes válidos.")

grupos = {
    "Todos": tiempos,
    "Trip": tiempos[tiempos["movement_type"].eq("trip")],
    "MicroTrip": tiempos[tiempos["movement_type"].eq("utrip")],
}
estadisticas = []
for nombre, grupo in grupos.items():
    if grupo.empty:
        continue
    grupo = grupo.sort_values("p50")
    pesos_acumulados = grupo["total_trips"].cumsum().to_numpy()
    umbrales = np.array([0.25, 0.50, 0.75]) * pesos_acumulados[-1]
    posiciones = np.searchsorted(pesos_acumulados, umbrales)
    q1, mediana, q3 = grupo["p50"].to_numpy()[posiciones]
    estadisticas.append(
        {
            "grupo": nombre,
            "minimo": grupo["p50"].min(),
            "q1": q1,
            "mediana": mediana,
            "q3": q3,
            "maximo": grupo["p50"].max(),
            "viajes": int(np.round(pesos_acumulados[-1], 0)),
        }
    )


resumen = pd.DataFrame(estadisticas)
base = alt.Chart(resumen).encode(
    x=alt.X("grupo:N", title=None, sort=list(grupos), axis=alt.Axis(labelAngle=0)),
    tooltip=[
        "grupo:N",
        "viajes:Q",
        "minimo:Q",
        "q1:Q",
        "mediana:Q",
        "q3:Q",
        "maximo:Q",
    ],
)

bigotes = base.mark_rule().encode(
    y=alt.Y(
        "minimo:Q",
        title="Tiempo de viaje (minutos)",
        axis=alt.Axis(
            values=list(
                range(0, int(np.ceil(resumen["maximo"].max() / 10)) * 10 + 1, 10)
            ),
            labelOverlap=False,
            labelBound=False,
        ),
    ),
    y2="maximo:Q",
)
caja = base.mark_bar(size=60).encode(
    y="q1:Q", y2="q3:Q", color=alt.Color("grupo:N", legend=None, sort=list(grupos))
)
linea_mediana = base.mark_tick(
    orient="horizontal", size=60, thickness=3, color="white"
).encode(y="mediana:Q")

alt.layer(bigotes, caja, linea_mediana).properties(
    width=450,
    height=500,
    title=alt.TitleParams(
        text="Tiempos de viaje por tipo de movimiento",
        subtitle="ODs sin tiempos de viaje se imputan como 3 horas. Análiis para la zona Core.",
        anchor="start",
        subtitleFontSize=12,
        subtitleColor="#666666",
    ),
)


alt.LayerChart(...)

In [30]:
# bigotes.display()
# para imprimir cuando no anda

### Porcentaje de viajes que podrían realizarse en transporte público
Calculamos, para cada zona de origen, qué proporción de los viajes podría completarse en transporte público en un máximo de 15, 30 y 45 minutos.

Luego repetimos el cálculo únicamente para los viajes clasificados como *Trips*, que suelen ser más largos y, por ello, podrían tener mayor probabilidad de requerir transporte público.

#### Todos los viajes

In [31]:
vmin = 0
vmax = 1
name = "Porcentaje de viajes en menos de "

for threshold in THRESHOLDS:
    zonas[name + "{} minutos".format(threshold)] = (
        ods_tt[ods_tt["p50"] <= threshold].groupby("origin")["total_trips"].sum()
        / ods_tt.groupby("origin")["total_trips"].sum()
    ).round(2)
    cols = ["geometry", name + "{} minutos".format(threshold)]
    m = simplify_for_web(zonas[zonas["tipo_zona"] == "Core"][cols]).explore(
        column=name + "{} minutos".format(threshold),
        cmap="Spectral",
        vmin=vmin,
        vmax=vmax,
        tiles=TILES,
        zoom_control=False,
    )
    display(m)

#### Viajes tipo Trips

In [32]:
vmin = 0
vmax = 1
name = "Porcentaje de viajes en menos de "
trips = ods_tt[ods_tt["movement_type"] == "trip"].copy()

for threshold in THRESHOLDS:
    zonas[name + "{} minutos".format(threshold)] = (
        trips[trips["p50"] <= threshold].groupby("origin")["total_trips"].sum()
        / trips.groupby("origin")["total_trips"].sum()
    ).round(2)
    cols = ["geometry", name + "{} minutos".format(threshold), "Viajes Generados Trip"]
    m = simplify_for_web(zonas[zonas["tipo_zona"] == "Core"][cols]).explore(
        column=name + "{} minutos".format(threshold),
        cmap="Spectral",
        vmin=vmin,
        vmax=vmax,
        tiles=TILES,
        zoom_control=False,
    )
    display(m)

In [33]:
vmin = 0
vmax = 1
name = "Porcentaje de viajes en menos de "
trips = ods_tt[ods_tt["movement_type"] == "trip"].copy()

for threshold in [45]:
    # no aclaro en el nombre de la columna si es trip o no.
    zonas["viajes en menos de {} minutos".format(threshold)] = (
        trips[trips["p50"] <= threshold].groupby("origin")["total_trips"].sum()
    )
    sc[name + "{} minutos".format(threshold)] = (
        (
            zonas.groupby("seccion_codigo")[
                "viajes en menos de {} minutos".format(threshold)
            ].sum()
        )
        / (zonas.groupby("seccion_codigo")["Viajes Generados Trip"].sum())
    ).round(2)

    cols = ["geometry", name + "{} minutos".format(threshold), "Viajes Generados Trip"]
    m = simplify_for_web(sc[cols]).explore(
        column=name + "{} minutos".format(threshold),
        cmap="Spectral",
        vmin=vmin,
        vmax=vmax,
        tiles=TILES,
        zoom_control=False,
    )
    display(m)

**Conclusiones de la sección**:
- El 75% de los viajes puede hacerse en menos de 30 minutos en trasporte público.
- El 75% de los viajes tipo trips puede hacerce en menos de 45 minutos en transporte público.
- En la zona central de montevideo más del 80% de los viajes tipo Trips podrían hacerse en menos de 45 minutos en transpote público. Vale la pena comparar estos resultados con el ridership actual. 
- Cuando nos alejamos de la zona central, hacia el norte, por la ruta 5, hasta Villa Felicidad, observamos niveles altos de accesibilidad similares a los descripto arriba. 

